# Pipeline KYC — Inventaire des documents & extraction du justificatif d'identité

**Objectif de ce notebook**

1. Dézipper `kyc_documents.zip` et repérer, dans chaque dossier client, les 5 documents utiles parmi tous
   ceux présents : `JUSTIFICATIF IDENTITE.PDF`, `JUSTIFICATIF DOMICILE.PDF`, `CONVENTION COMPTE.PDF`,
   `FATCA.PDF`, `CARTON SIGNATURE.PDF`.
2. Produire un **rapport d'inventaire** (CSV) indiquant, pour chaque client, quels documents sont présents.
3. Déterminer la **liste des clients cibles** : ceux pour lesquels `JUSTIFICATIF IDENTITE.PDF` existe.
4. Pour ces clients, **extraire automatiquement l'information** du `JUSTIFICATIF IDENTITE.PDF` (document
   scanné, potentiellement multi-page, multilingue, mal orienté ou de mauvaise qualité) à l'aide du modèle
   **Qwen3.5** (vision-langage) hébergé localement sur votre ModelHub Domino — aucune donnée ne sort de
   votre environnement.

**Hypothèses et choix retenus** (tout est regroupé en Partie 1 pour être ajusté facilement) :

- La liste des fichiers cibles contient `CARTON SIGNATURE.PDF` : l'énoncé initial mentionnait
  *"CARTON SIGNATUTE.PDF"*, probablement une coquille — corrigez `TARGET_FILES` si le nom réel diffère
  dans vos dossiers.
- Le `config.json` fourni indique `"model_type": "qwen3_5"`, c'est-à-dire l'architecture **Qwen3.5**
  (multimodale texte/image/vidéo). Ce nom diffère légèrement de celui du dossier ModelHub ("Qwen3.8"),
  probablement une convention de nommage interne : le code se base sur l'architecture réellement déclarée
  dans le config.json, pas sur le nom du dossier.
- La comparaison des noms de fichiers est **insensible à la casse** (les scans bancaires ont rarement une
  casse homogène), et chaque dossier client est parcouru **récursivement**.
- Seul `JUSTIFICATIF IDENTITE.PDF` est traité par le modèle dans ce notebook, comme demandé. Les 4 autres
  types de documents sont inventoriés ; le même patron de code (Parties 4 à 8) pourra leur être appliqué
  ensuite.
- Le traitement par lot est **reprenable** : chaque client traité produit un fichier JSON individuel, donc
  une interruption (crash, timeout) ne fait pas perdre le travail déjà effectué.
- Pour la conversion PDF → image, on utilise **PyMuPDF** plutôt que `pdf2image`/Poppler : aucune dépendance
  système (binaire externe) à installer, ce qui est plus robuste dans un environnement managé comme Domino.
- Ce notebook ne fige pas de numéros de version exacts pour `torch`/`transformers` (écosystème qui évolue
  vite) : il installe une **version plancher connue pour fonctionner avec Qwen3.5**, puis **enregistre les
  versions réellement installées** dans un fichier (`environnement_installe.txt`) pour la traçabilité.

## Partie 0 — Installation des librairies

Installez dans cet ordre : d'abord les librairies de fichiers/images (légères, sans risque), puis la pile
modèle (`transformers`, `accelerate`, `compressed-tensors`), et enfin `torch` — à adapter impérativement à
la version CUDA de votre environnement Domino (voir commentaire ci-dessous). Si `torch` est déjà préinstallé
dans votre image Domino (fréquent sur les environnements GPU), vous pouvez sauter cette ligne.

In [ ]:
# --- Traitement de fichiers / PDF / images (aucune dépendance système requise) ---
%pip install -q "pymupdf>=1.26.0"                   # rendu des pages PDF en images ; respecte la rotation déclarée dans le PDF
%pip install -q "pillow>=10.4.0"                    # manipulation d'images
%pip install -q "opencv-python-headless>=4.10.0"    # redressement (deskew) + contraste ; "headless" = pas de dépendance GUI/libGL
%pip install -q "numpy>=1.26.0"
%pip install -q "pandas>=2.2.0"                     # rapport d'inventaire, consolidation des résultats
%pip install -q "tqdm>=4.66.0"                      # barre de progression du traitement par lot

# --- Pile modèle : Qwen3.5 (vision-langage), quantifié FP8 ---
%pip install -q "transformers>=5.8.0"               # le model_type "qwen3_5" est supporté à partir de la 5.8 ; privilégiez la dernière version stable
%pip install -q "accelerate>=0.34.0"                # requis pour device_map="auto" (répartition automatique sur GPU)
%pip install -q "compressed-tensors>=0.7.0"         # requis pour décoder les poids quantifiés FP8 du checkpoint (cf. quantization_config, Partie 5)

# --- PyTorch : à adapter à VOTRE version CUDA (vérifiez avec `!nvidia-smi`) ---
# Décommentez et ajustez l'URL d'index si torch n'est pas déjà présent dans votre environnement Domino, par ex. :
# %pip install -q torch --index-url https://download.pytorch.org/whl/cu124

# --- Optionnel : accélère l'attention hybride (Gated DeltaNet) de Qwen3.5 ---
# Sans ces paquets, le modèle fonctionne normalement mais bascule automatiquement sur un mode de repli
# PyTorch plus lent. Leur compilation nécessite un toolchain CUDA correspondant exactement à votre torch :
# à tenter seulement si la vitesse d'inférence pose problème, sinon inutile de les installer.
# %pip install -q -U kernels
# %pip install -q causal-conv1d --no-build-isolation

## Imports groupés

Toutes les librairies utilisées dans ce notebook, importées une seule fois ici.

In [ ]:
# Bibliothèque standard
import os
import re
import io
import sys
import json
import time
import shutil
import zipfile
import logging
import platform
import subprocess
from pathlib import Path
from datetime import datetime

# Traitement de données / fichiers / images
import numpy as np
import pandas as pd
import cv2
import pymupdf
from PIL import Image
from tqdm.auto import tqdm

# Modèle
import torch
import transformers
from transformers import AutoProcessor, Qwen3_5ForConditionalGeneration

print("Toutes les librairies ont été importées avec succès.")

## Partie 1 — Configuration

Tous les paramètres modifiables du pipeline sont centralisés ici : chemins, liste des fichiers cibles,
options de prétraitement d'image. C'est le seul endroit à modifier pour adapter le notebook à votre
environnement exact.

In [ ]:
# ============================== CHEMINS ==============================
ZIP_PATH = Path("kyc_documents.zip")                       # <-- à adapter : emplacement réel du zip dans Domino
WORK_DIR = Path("kyc_pipeline_workdir")                     # tous les fichiers produits par ce notebook y seront rangés

RAW_EXTRACT_DIR  = WORK_DIR / "01_extraction_brute"          # dézippage complet et brut
FILTERED_DIR     = WORK_DIR / "02_documents_cibles"           # uniquement les 5 fichiers utiles, par client
RESULTS_DIR      = WORK_DIR / "03_resultats_identite"         # un fichier JSON par client traité (reprenable)

REPORT_PATH            = WORK_DIR / "rapport_inventaire_kyc.csv"
TARGET_CLIENTS_PATH    = WORK_DIR / "liste_clients_cibles.csv"
COMBINED_RESULTS_JSON  = WORK_DIR / "resultats_extraction_identite.json"
COMBINED_RESULTS_CSV   = WORK_DIR / "resultats_extraction_identite.csv"
LOG_PATH               = WORK_DIR / "pipeline_kyc.log"
ENV_SNAPSHOT_PATH      = WORK_DIR / "environnement_installe.txt"

MODEL_PATH = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.8-27B-FP8/main"  # chemin fourni

# ============================== FICHIERS CIBLES ==============================
# Comparaison insensible à la casse (voir normalize() en Partie 2) : la casse ci-dessous n'a pas besoin de
# correspondre exactement à celle des fichiers réels sur le disque.
TARGET_FILES = [
    "JUSTIFICATIF IDENTITE.PDF",
    "JUSTIFICATIF DOMICILE.PDF",
    "CONVENTION COMPTE.PDF",
    "FATCA.PDF",
    "CARTON SIGNATURE.PDF",     # "SIGNATURE" ; remplacez par "SIGNATUTE" si c'est réellement ce nom-là chez vous
]
IDENTITY_FILE = "JUSTIFICATIF IDENTITE.PDF"    # doit être un élément exact de TARGET_FILES ci-dessus

# ============================== PARAMÈTRES DE TRAITEMENT ==============================
PDF_RENDER_DPI               = 300     # résolution de rendu PDF -> image (300 = bon compromis qualité/vitesse)
ENABLE_ORIENTATION_CHECK     = True    # corrige les rotations franches (90/180/270°) via le modèle
ENABLE_SKEW_CORRECTION       = True    # corrige les légères inclinaisons (quelques degrés) via OpenCV
ENABLE_CONTRAST_ENHANCEMENT  = True    # améliore le contraste des scans de mauvaise qualité (CLAHE)
MAX_NEW_TOKENS_EXTRACTION    = 2048    # longueur max. de la réponse du modèle pour l'extraction structurée
FORCE_REPROCESS              = False   # True = retraite même les clients déjà traités (sinon reprise automatique)

# ============================== INITIALISATION ==============================
for d in (WORK_DIR, RAW_EXTRACT_DIR, FILTERED_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    handlers=[logging.FileHandler(LOG_PATH, encoding="utf-8"), logging.StreamHandler(sys.stdout)],
    force=True,  # évite les handlers dupliqués si la cellule est réexécutée
)
logger = logging.getLogger("kyc_pipeline")
logger.info("Configuration chargée. Répertoire de travail : %s", WORK_DIR.resolve())

# Remarque « protection des données » : les logs ne contiennent volontairement que des identifiants client
# et des statuts techniques — jamais les données personnelles extraites elles-mêmes.

Vérification de l'environnement (versions installées, GPU disponible) et sauvegarde d'un instantané des
versions réellement présentes, pour la traçabilité / l'audit (utile en contexte bancaire réglementé).

In [ ]:
print(f"Date d'exécution        : {datetime.now().isoformat(timespec='seconds')}")
print(f"Python                  : {sys.version.split()[0]} ({platform.system()} {platform.release()})")
print(f"PyTorch                 : {torch.__version__}")
print(f"Transformers            : {transformers.__version__}")
print(f"CUDA disponible         : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i} : {props.name} — {props.total_memory / 1e9:.1f} Go")
else:
    print("Aucun GPU détecté. L'inférence sur un modèle de cette taille sera très lente, voire impraticable, sur CPU.")

with open(ENV_SNAPSHOT_PATH, "w", encoding="utf-8") as f:
    f.write(subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
logger.info("Snapshot de l'environnement sauvegardé -> %s", ENV_SNAPSHOT_PATH)

## Partie 2 — Dézippage et inventaire des documents KYC

On dézippe `kyc_documents.zip`, puis pour **chaque dossier client**, on recherche les 5 fichiers cibles
(recherche récursive, insensible à la casse). Les fichiers trouvés sont copiés dans une arborescence propre
(`FILTERED_DIR/<client_id>/<nom_canonique>.PDF`), et leur présence/absence est consignée dans un tableau
qui sera sauvegardé en CSV.

In [ ]:
def normalize(name: str) -> str:
    """Normalise un nom de fichier pour une comparaison insensible à la casse et aux espaces superflus."""
    return name.strip().upper()

TARGET_FILES_NORM = {normalize(f): f for f in TARGET_FILES}

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"Archive introuvable : {ZIP_PATH.resolve()}. Vérifiez ZIP_PATH dans la cellule de configuration (Partie 1)."
    )

if RAW_EXTRACT_DIR.exists():
    shutil.rmtree(RAW_EXTRACT_DIR)
RAW_EXTRACT_DIR.mkdir(parents=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(RAW_EXTRACT_DIR)
logger.info("Archive dézippée -> %s", RAW_EXTRACT_DIR.resolve())

# Le zip peut soit contenir directement les dossiers clients, soit un unique dossier racine qui les englobe :
# on détecte automatiquement le bon niveau.
entries = list(RAW_EXTRACT_DIR.iterdir())
DATA_ROOT = entries[0] if (len(entries) == 1 and entries[0].is_dir()) else RAW_EXTRACT_DIR

client_folders = sorted(p for p in DATA_ROOT.iterdir() if p.is_dir())
logger.info("%d dossier(s) client détecté(s) sous %s", len(client_folders), DATA_ROOT)

In [ ]:
rows = []
for client_dir in tqdm(client_folders, desc="Inventaire des documents"):
    client_id = client_dir.name
    all_files = [p for p in client_dir.rglob("*") if p.is_file()]   # recherche récursive

    found = {}  # nom_cible_canonique -> chemin réel trouvé
    for f in all_files:
        canonical = TARGET_FILES_NORM.get(normalize(f.name))
        if canonical:
            found[canonical] = f

    row = {"client_id": client_id}
    for target in TARGET_FILES:
        row[target] = target in found
    row["nb_documents_cibles_trouves"] = len(found)
    row["dossier_complet"] = len(found) == len(TARGET_FILES)
    rows.append(row)

    if found:
        dest_dir = FILTERED_DIR / client_id
        dest_dir.mkdir(parents=True, exist_ok=True)
        for canonical_name, src_path in found.items():
            shutil.copy2(src_path, dest_dir / canonical_name)

inventory_df = pd.DataFrame(rows).set_index("client_id").sort_index()
inventory_df.to_csv(REPORT_PATH, encoding="utf-8-sig")

logger.info("Rapport d'inventaire sauvegardé -> %s", REPORT_PATH.resolve())
print(f"\n{len(inventory_df)} client(s) au total.")
for target in TARGET_FILES:
    print(f"  - {target:<32} présent chez {int(inventory_df[target].sum())} client(s)")
print(f"  - Dossiers complets (5/5)         : {int(inventory_df['dossier_complet'].sum())}")

inventory_df

## Partie 3 — Liste des clients cibles

Les **clients cibles** sont ceux pour lesquels `JUSTIFICATIF IDENTITE.PDF` a été trouvé : ce sont eux qui
seront traités dans la suite du notebook.

In [ ]:
target_clients = inventory_df.index[inventory_df[IDENTITY_FILE]].tolist()

pd.Series(target_clients, name="client_id").to_csv(TARGET_CLIENTS_PATH, index=False, encoding="utf-8-sig")
logger.info("%d client(s) cible(s) -> %s", len(target_clients), TARGET_CLIENTS_PATH.resolve())

print(f"{len(target_clients)} client(s) cible(s) sur {len(inventory_df)} :")
print(target_clients)

## Partie 4 — Prétraitement des scans

Trois fonctions indépendantes du modèle, appliquées à chaque page du `JUSTIFICATIF IDENTITE.PDF` :

1. **`pdf_to_images`** : convertit chaque page du PDF en image haute résolution. PyMuPDF applique déjà
   automatiquement la rotation éventuellement déclarée dans les métadonnées de la page (`/Rotate`).
2. **`correct_skew`** : corrige une **légère inclinaison** (quelques degrés), typique d'un document mal
   aligné sur le scanner, via une détection géométrique (OpenCV).
3. **`enhance_image`** : améliore le contraste d'un scan de mauvaise qualité (CLAHE).

Les **rotations franches** (90°/180°/270°, scan « à l'envers ») ne sont pas détectables de façon fiable
par la seule géométrie — elles sont corrigées en Partie 5, une fois le modèle chargé, car c'est lui qui
identifie le mieux le sens de lecture réel du texte (y compris en écriture manuscrite ou en arabe).

In [ ]:
def pdf_to_images(pdf_path: Path, dpi: int = PDF_RENDER_DPI) -> list:
    """Convertit chaque page d'un PDF (y compris scanné) en une image PIL RGB haute résolution."""
    images = []
    with pymupdf.open(pdf_path) as doc:
        for page in doc:
            pix = page.get_pixmap(dpi=dpi)
            img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
            images.append(img)
    return images

In [ ]:
def correct_skew(image: Image.Image, max_angle: float = 15.0) -> Image.Image:
    """Corrige une légère inclinaison (quelques degrés) via une détection géométrique du contenu texte.
    Ignore volontairement les angles > max_angle : au-delà, il s'agit probablement d'une rotation franche
    (90/180/270°), gérée séparément par detect_and_fix_orientation (Partie 5)."""
    arr = np.array(image)
    gray = cv2.bitwise_not(cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY))
    thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]
    coords = np.column_stack(np.where(thresh > 0))
    if len(coords) < 50:
        return image  # page quasi blanche : rien à corriger

    angle = cv2.minAreaRect(coords)[-1]
    angle = -(90 + angle) if angle < -45 else -angle
    if abs(angle) > max_angle or abs(angle) < 0.1:
        return image

    h, w = arr.shape[:2]
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    rotated = cv2.warpAffine(arr, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    return Image.fromarray(rotated)

In [ ]:
def enhance_image(image: Image.Image) -> Image.Image:
    """Améliore le contraste local d'un scan de mauvaise qualité (CLAHE sur le canal de luminance)."""
    arr = np.array(image)
    lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    result = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)
    return Image.fromarray(result)

## Partie 5 — Chargement du modèle Qwen3.5 (vision-langage) en local

Le `config.json` fourni indique `"model_type": "qwen3_5"` : il s'agit de l'architecture **Qwen3.5**, un
modèle nativement multimodal (texte / image / vidéo), supporté par `transformers` à partir de la version
5.8. Le checkpoint est quantifié **FP8** (probablement au format `compressed-tensors`, d'où sa dépendance
installée en Partie 0) : la cellule suivante inspecte le `quantization_config` réel pour confirmer le
format avant chargement.

On charge le modèle avec `dtype="auto"` (respecte la précision déjà présente dans le checkpoint, donc le
FP8) et `device_map="auto"` (répartition automatique sur le(s) GPU disponibles).

In [ ]:
config_path = Path(MODEL_PATH) / "config.json"
if not config_path.exists():
    raise FileNotFoundError(f"config.json introuvable à {config_path} — vérifiez MODEL_PATH (Partie 1).")

with open(config_path, encoding="utf-8") as f:
    model_config = json.load(f)

print("model_type            :", model_config.get("model_type"))
print("transformers_version  :", model_config.get("transformers_version"), "(version utilisée lors de la sauvegarde du modèle)")
print("quantization_config   :")
print(json.dumps(model_config.get("quantization_config", {}), indent=2, ensure_ascii=False))

In [ ]:
t0 = time.time()
logger.info("Chargement du modèle depuis %s ...", MODEL_PATH)

# Si cette ligne échoue avec une erreur "unrecognized model type" ou une ImportError sur
# Qwen3_5ForConditionalGeneration, votre version de transformers est probablement antérieure à la 5.8 :
# exécutez `%pip install -U transformers` puis redémarrez le kernel.
model = Qwen3_5ForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    dtype="auto",         # respecte la précision du checkpoint (FP8 via compressed-tensors)
    device_map="auto",    # répartit automatiquement sur le(s) GPU disponible(s)
)
processor = AutoProcessor.from_pretrained(MODEL_PATH)
model.eval()

logger.info("Modèle chargé en %.1fs.", time.time() - t0)
if torch.cuda.is_available():
    print(f"Mémoire GPU allouée après chargement : {torch.cuda.memory_allocated() / 1e9:.1f} Go")

Fonction générique d'appel au modèle, et détection/correction des rotations franches (90/180/270°) —
cette dernière a besoin du modèle chargé ci-dessus, d'où sa place ici plutôt qu'en Partie 4.

In [ ]:
def _ask_model(images: list, prompt: str, max_new_tokens: int) -> str:
    """Appel générique du modèle avec une ou plusieurs images + une consigne texte. Retourne la réponse brute."""
    content = [{"type": "image", "image": img} for img in images]
    content.append({"type": "text", "text": prompt})
    messages = [{"role": "user", "content": content}]

    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    trimmed = [out[len(inp):] for inp, out in zip(inputs["input_ids"], generated)]
    return processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()


ORIENTATION_PROMPT = (
    "Regarde cette image de document scanné. Le texte est-il actuellement à l'endroit et horizontal ? "
    "Sinon, de combien de degrés faut-il la faire pivoter DANS LE SENS DES AIGUILLES D'UNE MONTRE pour "
    "qu'il le devienne : 90, 180 ou 270 ? Réponds uniquement par un seul chiffre parmi 0, 90, 180, 270 "
    "— aucun autre mot."
)

def detect_and_fix_orientation(image: Image.Image) -> Image.Image:
    """Détecte une rotation franche (90/180/270°) via le modèle, et corrige l'image en conséquence."""
    if not ENABLE_ORIENTATION_CHECK:
        return image
    try:
        answer = _ask_model([image], ORIENTATION_PROMPT, max_new_tokens=8)
        match = re.search(r"\b(90|180|270|0)\b", answer)
        rotation_needed = int(match.group(1)) if match else 0
    except Exception as e:
        logger.warning("Détection d'orientation impossible (%s) — image conservée telle quelle.", e)
        return image
    if rotation_needed == 0:
        return image
    # PIL.Image.rotate() tourne dans le sens ANTIhoraire pour un angle positif -> signe négatif pour un pivot horaire
    return image.rotate(-rotation_needed, expand=True)

## Partie 5bis — Test de sanité

Avant de lancer le traitement complet, on vérifie sur **un seul client** que le modèle charge
correctement, « voit » l'image, et produit une réponse cohérente. Cela évite de découvrir un problème
(chemin, prompt, mémoire GPU...) seulement après avoir attendu la fin d'un traitement par lot de plusieurs
dizaines de minutes.

In [ ]:
if not target_clients:
    print("Aucun client cible — vérifiez le rapport d'inventaire (Partie 2) avant de continuer.")
else:
    sample_client = target_clients[0]
    sample_pages = pdf_to_images(FILTERED_DIR / sample_client / IDENTITY_FILE)
    print(f"Client de test : {sample_client} — {len(sample_pages)} page(s) détectée(s).")

    sample_page = detect_and_fix_orientation(sample_pages[0])
    sample_page = correct_skew(sample_page)
    sample_page = enhance_image(sample_page)
    display(sample_page)  # aperçu de la page telle que le modèle va la recevoir

    quick_answer = _ask_model(
        [sample_page],
        "En une phrase : quel type de document est visible sur cette image, et le texte te semble-t-il "
        "net et bien orienté ?",
        max_new_tokens=100,
    )
    print("\nRéponse du modèle :", quick_answer)
    print("\nSi cette réponse est cohérente, vous pouvez passer à la suite en toute confiance.")

## Partie 6 — Extraction structurée depuis `JUSTIFICATIF IDENTITE.PDF`

Toutes les pages du document (recto/verso, etc.) sont envoyées **ensemble** en une seule requête au
modèle, qui peut ainsi croiser les informations entre pages. Le prompt :

- couvre les documents multilingues (français / arabe / anglais / tamazight / mélange), fréquents sur les
  pièces d'identité algériennes souvent bilingues ;
- demande une sortie **JSON strict**, avec un champ `texte_brut_ocr` (transcription complète, en
  repli/traçabilité) et un champ `champs_incertains` (liste des champs à faible confiance, pour cibler la
  revue manuelle) ;
- interdit explicitement au modèle de traduire les noms propres ou d'inventer une valeur absente.

Adaptez librement le schéma ci-dessous aux champs réellement exigés par votre équipe conformité.

In [ ]:
EXTRACTION_PROMPT = """Tu es un système expert en extraction de données à partir de documents d'identité (carte nationale d'identité, passeport, permis de conduire, titre de séjour, etc.) dans un contexte bancaire de connaissance client (KYC).

Les images fournies sont TOUTES les pages d'un même document, appartenant à un même client (par exemple le recto et le verso d'une carte d'identité). Le document peut être rédigé en français, en arabe, en anglais, en tamazight, ou dans un mélange de ces langues (les cartes d'identité algériennes sont souvent bilingues arabe/français). Le scan peut être de mauvaise qualité, incliné, flou, ou comporter de l'écriture manuscrite.

Analyse l'ensemble des pages fournies et réponds UNIQUEMENT avec un objet JSON valide (sans texte avant ou après, sans balises markdown), respectant exactement ce schéma :

{
  "type_document": string ou null,
  "nom": string ou null,
  "prenom": string ou null,
  "nom_arabe": string ou null,
  "prenom_arabe": string ou null,
  "date_naissance": string ou null,
  "lieu_naissance": string ou null,
  "sexe": string ou null,
  "nationalite": string ou null,
  "numero_document": string ou null,
  "numero_identification_nationale": string ou null,
  "date_delivrance": string ou null,
  "date_expiration": string ou null,
  "autorite_delivrance": string ou null,
  "adresse": string ou null,
  "langues_detectees": string ou null,
  "champs_incertains": [string],
  "texte_brut_ocr": string
}

Règles impératives :
- Ne traduis JAMAIS les noms propres, adresses ou numéros : recopie-les exactement comme ils apparaissent.
- Si une information est absente ou illisible, mets null plutôt que d'inventer une valeur.
- Liste dans "champs_incertains" le nom de chaque champ que tu n'es pas sûr d'avoir correctement lu.
- "texte_brut_ocr" doit contenir une transcription complète de tout le texte visible, page par page.
"""

print(f"Longueur du prompt d'extraction : {len(EXTRACTION_PROMPT)} caractères.")

In [ ]:
def extract_json_from_text(text: str) -> dict:
    """Extrait le premier objet JSON valide d'une réponse de modèle, même si celle-ci contient des balises
    ```json ... ``` ou du texte superflu avant/après malgré la consigne."""
    cleaned = re.sub(r"^```(?:json)?", "", text.strip()).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    start = cleaned.find("{")
    if start == -1:
        raise ValueError("Aucun objet JSON trouvé dans la réponse du modèle.")
    depth = 0
    for i, ch in enumerate(cleaned[start:], start=start):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return json.loads(cleaned[start:i + 1])
    raise ValueError("Objet JSON incomplet dans la réponse du modèle.")

In [ ]:
def process_one_client(client_id: str) -> dict:
    """Traite le JUSTIFICATIF IDENTITE.PDF d'un client : conversion en images, prétraitement, appel au
    modèle, structuration du résultat. Ne lève jamais d'exception : les échecs sont capturés et consignés
    dans le résultat retourné (statut="echec"), pour ne jamais interrompre le traitement par lot."""
    t0 = time.time()
    result = {"client_id": client_id, "statut": "echec", "erreur": None, "nb_pages_traitees": 0}
    pdf_path = FILTERED_DIR / client_id / IDENTITY_FILE

    try:
        if not pdf_path.exists():
            raise FileNotFoundError(f"Fichier introuvable : {pdf_path}")

        pages = pdf_to_images(pdf_path)
        if not pages:
            raise ValueError("Le PDF ne contient aucune page exploitable.")

        processed_pages = []
        for page_img in pages:
            img = detect_and_fix_orientation(page_img) if ENABLE_ORIENTATION_CHECK else page_img
            img = correct_skew(img) if ENABLE_SKEW_CORRECTION else img
            img = enhance_image(img) if ENABLE_CONTRAST_ENHANCEMENT else img
            processed_pages.append(img)

        raw_answer = _ask_model(processed_pages, EXTRACTION_PROMPT, MAX_NEW_TOKENS_EXTRACTION)
        structured = extract_json_from_text(raw_answer)

        result.update(structured)
        result["nb_pages_traitees"] = len(processed_pages)
        result["statut"] = "succes"

    except Exception as e:
        result["erreur"] = str(e)
        logger.error("[%s] échec de l'extraction : %s", client_id, e)

    result["duree_secondes"] = round(time.time() - t0, 1)
    return result

## Partie 7 — Traitement par lot des clients cibles

Chaque client traité produit immédiatement son fichier `RESULTS_DIR/<client_id>.json`. Si le notebook est
interrompu puis relancé, les clients déjà traités sont automatiquement ignorés (sauf `FORCE_REPROCESS =
True` en Partie 1) : le traitement reprend là où il s'était arrêté.

In [ ]:
for client_id in tqdm(target_clients, desc="Extraction JUSTIFICATIF IDENTITE"):
    out_path = RESULTS_DIR / f"{client_id}.json"
    if out_path.exists() and not FORCE_REPROCESS:
        continue  # déjà traité lors d'une exécution précédente

    result = process_one_client(client_id)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    logger.info("[%s] statut=%s (%ss)", client_id, result["statut"], result["duree_secondes"])

print(f"\nTerminé. Résultats individuels disponibles dans {RESULTS_DIR.resolve()}")

## Partie 8 — Consolidation des résultats

On rassemble tous les fichiers JSON individuels en un seul export JSON et un export CSV (pratique pour
Excel / votre équipe conformité), puis on affiche un résumé et la liste des éventuels échecs à examiner
manuellement.

In [ ]:
all_results = []
for f in sorted(RESULTS_DIR.glob("*.json")):
    with open(f, encoding="utf-8") as fh:
        all_results.append(json.load(fh))

if not all_results:
    print("Aucun résultat à consolider pour le moment (le traitement par lot n'a peut-être pas encore été exécuté).")
else:
    with open(COMBINED_RESULTS_JSON, "w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)

    results_df = pd.json_normalize(all_results)
    results_df.to_csv(COMBINED_RESULTS_CSV, index=False, encoding="utf-8-sig")

    nb_succes = int((results_df["statut"] == "succes").sum())
    nb_echec = int((results_df["statut"] == "echec").sum())
    print(f"{nb_succes} succès / {nb_echec} échec(s) sur {len(results_df)} client(s) cible(s) traité(s).")

    if nb_echec:
        print("\nClients en échec (à vérifier manuellement) :")
        print(results_df.loc[results_df["statut"] == "echec", ["client_id", "erreur"]].to_string(index=False))

    logger.info("Consolidation terminée -> %s / %s", COMBINED_RESULTS_JSON, COMBINED_RESULTS_CSV)

results_df.head(10) if all_results else None

## Conclusion & prochaines étapes

- Le rapport d'inventaire (`rapport_inventaire_kyc.csv`) couvre les **5** types de documents ; seule
  l'extraction (Parties 4 à 8) se limite pour l'instant à `JUSTIFICATIF IDENTITE.PDF`, comme demandé.
- Pour traiter un autre type de document (`JUSTIFICATIF DOMICILE.PDF`, `FATCA.PDF`...), dupliquez les
  Parties 6 à 8 en adaptant `IDENTITY_FILE` et le schéma JSON du prompt aux champs propres à ce document.
- Avant un passage à l'échelle, validez la qualité de l'extraction sur un échantillon (10-20 clients) en
  comparant manuellement `resultats_extraction_identite.csv` aux documents sources, et affinez le prompt
  si nécessaire — en particulier pour les champs qui reviennent souvent dans `champs_incertains`.
- Vérifiez que ce traitement s'inscrit bien dans votre politique de confidentialité des données / conformité
  RGPD (ou équivalent local), notamment sur la durée de conservation des fichiers intermédiaires générés
  (`kyc_pipeline_workdir/`).